# Analysis Notebook for Nexus System v12 (v1.0.0)

Focus on v12 was the compilation of the full nexus system using 4 nexus segments and incorporating the Judge and Handler nodes.

## Abstract:

### The issues:

Initially referenced in v10 analysis notebook: Artificial Intelligence (AI) currently faces three major issues:
-	Size: AI models are incredibly large (up to a couple billion parameters sometimes). This can require massive datacenters with specific and expensive hardware, incurring major energy costs.
-	Transparency: Little is known about how AI makes its computations. It is frequently referred to as a black box.
-	Limited modularity: While Ais can be configured with different settings (ex. amount of hidden layers), it is difficult to isolate issues or integrate different models together.


### Intent:

The goal of this project is to find a way to help counter these issues securely. The “Nexus” system focuses on 3 areas:
-	Maintaining accuracy despite having less nodes
-	Improve traceability of computations
-	Maximize modularity, potentially integrating various types of AI’s together


### The Nexus:

Using a specialized 'Judge' node, I will route specific queries to different 'segments,' each with their own specialty in a specific type of data. The selected segments will then load their nodes and run the computation. The returns of these segments will be merged into a final prediction. Eventually, this process may be chained together with different specialist groups across a pipeline, but that exceeds the original intent. 
This approach will theoretically address the three issues previously discussed. 
-	Size: By dynamically loading specialist segments, I hope to lower the amount of time and memory it takes to run a computation. Why load 100 nodes if only 25 is needed? Additionally, I hope to be able to 'prune' unnecessary nodes. If a node is needless space, then I will get rid of it.
-	Transparency: I will create a custom segment type to allow better tracing of computations (both visually and statistically). This may potentially reveal more about how AI works.
-	Modularity: By having various specialist segments, the modularity of this framework will be guaranteed. Why not train segments separately in parallel? With this framework it would be possible to swap different segments as needed, allowing for various specializations to be configured as needed. 
Overall, we will be sacrificing the raw efficiency and simplicity of modern neural networks to have a potentially flexible, specialized, and interpretable system.


### Current Focus: The full system


The v12 iteration focuses on the development of the full system. As I have previously developed the code for the nexus segment in v10, I have built up the full system integration. It has revealed some strengths and weaknesses behind this implementation.

### Observed Cons:

Currently I face these issues:

Training difficulty:
As previously observed, this will be a very difficult issue to address. Like before there are the segment specific issues:
- Sparse activation: It is difficult to update all the weights needed when only a subset of the nodes learn
- Overfitting: The narrower domain leads individual segments to have poor generalization skills
- Time complexity: The segment training involves both position and node weights. This makes time for training much more costly compared to a traditional model
- Node tracking: Tracking the nodes to determine which to prune is expensive
However, there are now some system specific training difficulties:


Risk:
Like previously observed:
- If the Judge node miscalculates or misroutes, the error could cascade and sharply impact performance
- If the Judge gets too complex then it could become a computationally heavy bottleneck 
- Data formatting: It would be necessary to have standardized inputs and outputs for each computation. For certain AI types, this could become costly to align
- Curse of Dimensionality: A feature of this framework is changing the number of dimensions used for computation as its graph based. Increasing the amount of dimensions can cause exponential growth in cost.


Implementation limitations:

Like before, careful practice can minimize the impacts of these cons, but compared to traditional model pipelines, these are much more difficult to counteract. Careful design is required when implementing this implementation.

### Observed Pros:

### Potential Applications:

I have observed these potential applications:

Claude (Claude Sonnet 4.6) observed applications:

## The various nodes/components:

Referenced previously in the v10 analysis, here is a recap:


#### Segment Nodes:
- Splitter Node: This node generates the initial signals and distributes that information to its nearest p% of processing nodes. It also calculates a feature relevance score for each feature to help give emphasis of certain features which may be relevant to the given dataset.

- Processing Node: These nodes are designed to be the main computational node in this neural net. Each processing node has a queue to store signals as they come in. These nodes use the current input as a feature, other feature weights, distance-based scaling and clamping to update a given prediction. They update a given prediction instead of generating a new one. To forward signals they save their signature in the signals visited nodes, and for each connected node calculate a weight for each node before randomly selecting one to forward to. This places emphasis on reviewer nodes and nodes further from the origin than others. In the case of a dead end, the signal will reset the list of the last 3 visited nodes. This list is a hard buffer preventing the visit of the past 3 visited nodes, but in the case of a dead end this list is reset allowing for back tracking of a given signal.

- Reviewer Node: The reviewer node is strictly mathematical without learning. It waits until each signal is received before reviewing the signals with inverse variance weighting so that signals with less variance (a more direct path) have higher weight. With this, it uses weighted sums which becomes the final prediction for that node.

#### Universal nodes/components:
- Signal: The signal is the key component in the processing. It is simply a class that saves the current prediction, variance, position, etc. 

- Preprocessing node: This node is implemented in segment handler as it is required for all preprocessing needed. When computing for a specific input, it will tokenize the categorical data and normalize the lists, tuples, or series. When handling full databases, it drops the removable columns, takes care of missing values, and then treats each row as an input to generate the database. Scalars themselves are not normalized.

#### Now-implemented System Nodes:
- Judge node: This node is designed to generate clusters from the data. This could allow for segments to specialize in a manner akin to Mixture of Experts models. Inputs will be distributed to needed segments accordingly.
- Handler node: Like a segment Reviewer node except the handler is designed to handle all reviewer node returns. However, this node uses a count of predictions, not weights.




## The Key Settings

Same as v10, there are segment specific settings:
- max x: This is the maximum x and y for the segment graph. It is the radius of the segment's arc and directly influences how many nodes can be loaded (as a max limit). More than that, it works with the power of dimensions and the density to directly influence how many nodes are loaded. 

 - target: This is the target column for the predictions. In the case of this demo, it is the exam score.

- connection percentage: This impacts how many nodes each node can connect to. For the splitter, it is a percentage based on the total amount of nodes, but for processing nodes it represents a percentage of 50 (so there can be a max of 50 connections given a high enough percentage). By default, I have this set to .1.

- density: This is the number of nodes that will attempt to be loaded as a percentage of the total area given the max x.

- dimensions: This is how many dimensions will be loaded. Higher dimensions will provide exponentially more nodes but will also be affected by the curse of dimensionality. By default, this is set to 2.

- classification: This is used for logging purposes. 4 is info level (minimal logs), 3 is warning info, 2 is error info, and 1 is full debug. By default, this is 4.

- segment ID: This is an identifier for the current segment. While its use is limited right now, it will have use when a full system is developed.

- Nexseg file: This is the file path to save the segment information. This allows the best segments to be saved and loaded in the future (untested).

- Dataset CSV: This is the path for the dataset. In this case it is Exam_Score_Prediction.csv.

Splitter: 

- Grad clip: This clamps the per feature relevance gradient to [-1,1]

- Weight clip: this clamps the per feature relevance weights to [-5, 5]

Processing:

- delta clip: This clamps each nodes scaled delta (its contribution) before it is added to the active prediction. By default, this is [-10, 10].

- pred clip: This clamps the running prediction to prevent prediction explosion and overflow. By default, this is [-1000000, 1000000].

- grad clip: This clamps the weight and position gradients. By default, this is [-1, 1].

- Weight clip: This clamps all the learned feature weights. By default, this is [-5, 5].

- min connections: This guarantees each processing has a certain number of connections regardless of connection percentage. By default, this is 3.

- position penalty: This is used to regularize the amount of movement. When the position gradient is calculated, this penalty applies based on the distance from the nodes original position preventing excessive movement and structural collapse. By default, this is .1.

- reviewer bonus: This multiplies a reviewer node's routing weight by a constant, so signals are biased towards the reviewer. By default, this is 3.

Training settings: 

- Weight LR: The weight learning rate is the base rate for how much the node feature weights can update. By default, this is .01.

- Position LR: The position learning rate is the base rate for how much the nodes can move. By default, this is set to .005 to prevent excessive movement.

- Splitter LR: The splitter learning rate is the base rate for the splitters feature relevance scoring system. By default, this is .01.

- LR decay: The learning rate decay decreased the learning rates for the above three over each epoch. By default, this is .85.

- Max Position Step: This is the hard cap on how far a node can move per update regardless of the gradient. It is a final safety. By default, this is 2.

- Convergence Threshold: This determines if early stopping is needed. Training will stop if the change in loss between consecutive epochs is below this threshold. By default, it is 1e-4.

- Plateau Threshold: Since reconnections can generate a lot of noise and interfere with learning, this threshold exists. If the range of the last 3 test errors are under this threshold percentage, then the plateau status is implemented (refer to training procedure below). By default, this is set to 3%.

- Plateau Position LR Scale: When plateau status is enabled, the position learning rate is multiplied by this. By default, this is set to .01. 

There are also new settings to address:
- Judge Iters:
- Judge Min Clusters:
- Judge Max clusters:
- Selection Percentage:
- Training Mode:
- Aggregation Mode:
- Seg Colors:

## The procedure for a segment:

This is a recap of the v10 explanation for how a segment handler works.


Inference in `segmentInfer()` is structurally identical to the training forward pass but stripped of all gradient machinery. No weights, positions, or feature relevance scores are modified. The method takes a pre-processed input dict (the caller's responsibility to prepare via `PreProcessingNode`) and returns a list of per-reviewer prediction reports.

---

##### 1. State Reset

Before any signals are generated, all residual state from prior runs is cleared:
- Every `ProcessingNode` has its `signal` and `signal_queue` set to `None`/`[]` via `clear_signals()`
- Every `ReviewerNode` has its `signals` list emptied

This ensures no stale signal from a previous inference contaminates the current pass.

---

##### 2. Signal Generation and Dispatch

The `SplitterNode.process()` call computes the current feature relevance scores from its learned `signal_weights` and generates one `Signal` per connected node. Each signal is initialized with:
- `prediction = 0.0`
- `input` — the pre-processed feature dict
- `feature_relevance` — the splitter's current `signal_weights` (learned during training)
- `variance = 0.0`
- `signal_life = (max_x² × 0.8)`
- `path_contributions = {}` — allocated but never written to during inference

Signals are then dispatched one-to-one to the splitter's connected processing nodes via `node.receive_signal(signal)`.

---

##### 3. Processing Loop

The main loop runs for up to 500 iterations. Each iteration walks every `ProcessingNode`. If a node holds a signal, it calls `process_signal()` — the inference-mode counterpart to `train_process_signal()`. The math is identical:

```
weighted_sum = prediction × w_pred
             + Σ (value_i × w_i × rel_i)   for each feature i

scaled_delta = clip(weighted_sum / (1 + dist_to_origin), −10, 10)

signal.prediction += scaled_delta
signal.variance   += |scaled_delta|
```

The critical difference from training: **`path_contributions` is never populated**. There is nothing to backpropagate through, so the contribution recording step is simply absent. `activation_count` is also not incremented during inference.

After processing, the node calls `forward_signal()`, which is unchanged from training — the same stochastic routing with outward alignment bias, the reviewer bonus (`×3.0`), and the `recent_visited` anti-cycle buffer of size 3.

The loop terminates when every signal is either expired (`signal_life ≤ 0`) or has been collected by a `ReviewerNode` (`signal.collected == True`). A hard cap of 500 iterations prevents infinite loops if signals get trapped.

---

##### 4. Reviewer Aggregation

When a signal reaches a `ReviewerNode`, `receive_signal()` resets its `signal_life` to 5 and marks it `collected = True`. After the loop ends, each reviewer calls `review_signals()`, which computes the same inverse-variance weighted average used in training:

```
w_i    = 1 / max(variance_i, 1e-9)
output = Σ(prediction_i × w_i) / Σ(w_i)
```

If no valid signals reached a reviewer (all expired before arrival), `review_signals()` returns `None` and that reviewer produces no report.

---

##### 5. Return Value

`segmentInfer()` returns a list of report dicts, one per reviewer that produced a valid prediction:

```python
{'id': segment_id, 'prediction': float, 'reviewer_position': tuple}
```

In the full system this list would be passed to a `HandlerNode`, which aggregates across all segments. As that is not yet implemented, the analysis notebook uses a simple average across all reviewer predictions as a stand-in final output.

For a demo on the segment training and inference procedure please refer to the v10 analysis notebook. 

## DEMO:

This demo will focus on the full system implentation with training and inference. Similar to the segment demo, it will feature pre-training, training, and post training procedures.

### The training data:
Like before, I am using a dataset from Kaggle called Exam Score Prediction. It is a csv with the following information:

import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
data = pd.read_csv('Exam_Score_Prediction.csv')

print("\nFirst 5 rows of the dataset:")
data.head()



print("\nDataset Information:")
data.info()

### Pre training inference:
I will demonstrate the results when running this segment using a random input row before training is done. The result will be random noise but will serve as a point of comparison.

### Training:
Below is where I demonstrate the training of the model with 20 epochs.

### Post Test Inference and Summary

## Comparing to other models

## Next Plans

This project will be proceed to its implementation phase. I will use this project in the various ways to test its different potential applications. The next steps will be:
- Computation analysis: A tool that will show how predictions were generated for a given input
- Simmulated attack demo: Use this architecture with reinforcement learning to simulate a network attack that the blue team (this architecture) will learn to defend against. 